In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    auc,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. FONCTION COMMUNE D'EXTRACTION DES FEATURES
# ==============================================================================
def extract_session_features(df_telemetry):
    """Calcule les indicateurs physiques par session de charge."""
    df_telemetry = df_telemetry.copy()
    df_telemetry["timestamp"] = pd.to_datetime(df_telemetry["timestamp"])

    features_list = []
    for (station_id, sess_id), grp in df_telemetry.groupby(
        ["charge_point_id", "session_id"]
    ):
        duration_min = (
            grp["timestamp"].max() - grp["timestamp"].min()
        ).total_seconds() / 60.0
        if duration_min < 5:
            continue

        delta_t = (
            grp["temp_internal_pcb_c"].iloc[-1]
            - grp["temp_internal_pcb_c"].iloc[0]
        )
        slope_t = delta_t / max(1.0, duration_min)
        thermal_idx = (
            (grp["temp_internal_pcb_c"] - grp["temp_ambient_c"])
            / (grp["power_active_kw"] + 0.5)
        ).mean()

        charging_rows = grp[grp["current_l1_a"] > 5.0]
        if len(charging_rows) > 0 and (charging_rows["voltage_l2_v"] > 50).any():
            v_drop_l1 = (
                charging_rows["voltage_l2_v"] - charging_rows["voltage_l1_v"]
            ).mean()
        else:
            v_drop_l1 = 0.0

        is_degraded = (
            grp["is_degraded_state"].max()
            if "is_degraded_state" in grp.columns
            else np.nan
        )

        features_list.append(
            {
                "charge_point_id": station_id,
                "session_id": sess_id,
                "mean_power_kw": grp["power_active_kw"].mean(),
                "max_temp_pcb": grp["temp_internal_pcb_c"].max(),
                "temp_slope": slope_t,
                "v_drop_l1_vs_l2": v_drop_l1,
                "thermal_resistance_idx": thermal_idx,
                "is_degraded_session": is_degraded,
            }
        )

    return pd.DataFrame(features_list)


# ==============================================================================
# 2. CHARGEMENT ET EXTRACTION DES FEATURES (TRAIN & TEST)
# ==============================================================================
features_cols = [
    "mean_power_kw",
    "max_temp_pcb",
    "temp_slope",
    "v_drop_l1_vs_l2",
    "thermal_resistance_idx",
]

# Jeu d'entraînement (enrichi / balance 87-13)
print("Traitement du jeu d'entraînement...")
df_tel_train = pd.read_csv("hager_telemetry_multistation_extended.csv")
df_train_sessions = extract_session_features(df_tel_train)

# Jeu de test réaliste (conditions réelles 99.5 - 0.5)
print("Traitement du jeu de test réaliste...")
df_tel_realistic = pd.read_csv("hager_telemetry_realistic_distribution.csv")
df_fleet_realistic = pd.read_csv("hager_fleet_inventory_realistic.csv")
df_new_sessions = extract_session_features(df_tel_realistic)

# ==============================================================================
# 3. FIT DU SCALER ET DU MODÈLE SUR LE TRAIN
# ==============================================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train_sessions[features_cols])

# Isolation Forest entraîné sur l'espace d'entraînement
iso = IsolationForest(contamination=0.13, random_state=42, n_estimators=150)
iso.fit(X_train)

# ==============================================================================
# 4. INFÉRENCE SUR LE JEU RÉALISTE (SANS FUITE)
# ==============================================================================
X_test = scaler.transform(df_new_sessions[features_cols])

# Score continu : plus le score est élevé, plus le profil est anormal
test_scores = -iso.decision_function(X_test)
df_new_sessions["anomaly_score"] = test_scores

# Calibrage du seuil pour l'opérationnel :
# Sur le jeu réel, 0.5 % des sessions sont anormales. On fixe le seuil au 99.5e percentile.
SEUIL_OPERATIONNEL = np.percentile(test_scores, 99.5)
df_new_sessions["is_anomaly_pred"] = (
    test_scores >= SEUIL_OPERATIONNEL
).astype(int)

# ==============================================================================
# 5. ÉVALUATION NIVEAU 1 : SESSIONS (TEST RÉALISTE)
# ==============================================================================
y_true_sess = df_new_sessions["is_degraded_session"]
y_pred_sess = df_new_sessions["is_anomaly_pred"]

roc_sess = roc_auc_score(y_true_sess, test_scores)
prec_vals, rec_vals, _ = precision_recall_curve(y_true_sess, test_scores)
pr_auc_sess = auc(rec_vals, prec_vals)
cm_sess = confusion_matrix(y_true_sess, y_pred_sess)

print("\n" + "=" * 60)
print("1. RÉSULTATS SESSIONS SUR DISTRIBUTION RÉALISTE (99.5 / 0.5)")
print("=" * 60)
print(f"ROC-AUC (indépendant du seuil) : {roc_sess:.3f}")
print(f"PR-AUC  (clé sur déséquilibre) : {pr_auc_sess:.3f}")
print(
    f"Balanced Accuracy              : {balanced_accuracy_score(y_true_sess, y_pred_sess):.3f}\n"
)
print("Matrice de confusion (Sessions) :")
print(f"  Vrais Négatifs (VN) : {cm_sess[0,0]} | Faux Positifs (FP) : {cm_sess[0,1]}")
print(f"  Faux Négatifs  (FN) : {cm_sess[1,0]} | Vrais Positifs (VP) : {cm_sess[1,1]}\n")
print(
    classification_report(
        y_true_sess,
        y_pred_sess,
        target_names=["Nominale", "Dégradée"],
        digits=3,
    )
)

# ==============================================================================
# 6. ÉVALUATION NIVEAU 2 : PARC DE BORNES RÉALISTE
# ==============================================================================
# Agrégation au niveau de la borne
station_summary = (
    df_new_sessions.groupby("charge_point_id")
    .agg(
        total_sessions=("session_id", "count"),
        nb_anomalies_pred=("is_anomaly_pred", "sum"),
        taux_anomalies=("is_anomaly_pred", "mean"),
    )
    .reset_index()
)

station_eval = pd.merge(
    station_summary, df_fleet_realistic, on="charge_point_id"
)

# Seuil de décision : une borne réelle ayant plusieurs anomalies répétées est suspecte
# Avec 0.5 % de bruit de fond, un taux > 5 % d'anomalies sur une borne indique un problème
SEUIL_ALERTE_BORNE = 0.05
station_eval["has_fault_pred"] = (
    station_eval["taux_anomalies"] > SEUIL_ALERTE_BORNE
).astype(int)

y_true_borne = station_eval["has_fault"]
y_pred_borne = station_eval["has_fault_pred"]
cm_borne = confusion_matrix(y_true_borne, y_pred_borne)

print("=" * 60)
print("2. RÉSULTATS DÉCISION PARC SUR BORNES RÉALISTES")
print("=" * 60)
print(
    f"ROC-AUC Borne : {roc_auc_score(y_true_borne, station_eval['taux_anomalies']):.3f}\n"
)
print("Matrice de confusion (Bornes) :")
print(
    f"  Bornes saines confirmées       : {cm_borne[0,0]} / {cm_borne[0,0] + cm_borne[0,1]}"
)
print(f"  Fausses alertes (déplacements) : {cm_borne[0,1]}")
print(
    f"  Pannes bien identifiées        : {cm_borne[1,1]} / {cm_borne[1,0] + cm_borne[1,1]}"
)
print(f"  Pannes manquées                : {cm_borne[1,0]}\n")
print(
    classification_report(
        y_true_borne,
        y_pred_borne,
        target_names=["Saine", "En panne"],
        digits=3,
    )
)

Traitement du jeu d'entraînement...
Traitement du jeu de test réaliste...

1. RÉSULTATS SESSIONS SUR DISTRIBUTION RÉALISTE (99.5 / 0.5)
ROC-AUC (indépendant du seuil) : 1.000
PR-AUC  (clé sur déséquilibre) : 1.000
Balanced Accuracy              : 0.999

Matrice de confusion (Sessions) :
  Vrais Négatifs (VN) : 3312 | Faux Positifs (FP) : 4
  Faux Négatifs  (FN) : 0 | Vrais Positifs (VP) : 13

              precision    recall  f1-score   support

    Nominale      1.000     0.999     0.999      3316
    Dégradée      0.765     1.000     0.867        13

    accuracy                          0.999      3329
   macro avg      0.882     0.999     0.933      3329
weighted avg      0.999     0.999     0.999      3329

2. RÉSULTATS DÉCISION PARC SUR BORNES RÉALISTES
ROC-AUC Borne : 1.000

Matrice de confusion (Bornes) :
  Bornes saines confirmées       : 48 / 48
  Fausses alertes (déplacements) : 0
  Pannes bien identifiées        : 2 / 2
  Pannes manquées                : 0

              p